In [4]:
# Community-verification/util/io.py をimportするためのコード
# io.pyを編集した場合に、Jupyter Notebook上で再読み込みするために必要
import importlib
import os
import sys
sys.path.append(os.path.abspath("..")) 
from util import io
importlib.reload(io)

<module 'util.io' from '/Users/masaaki/dev/community-detection-social-networks/Community-verification/util/io.py'>

### LFR generation

In [17]:
import networkx as nx
import numpy.random as random

def generate_lfr_benchmark_graph(
  n: int, tau1: float, tau2: float, mu: float,
  average_degree: int, min_community: int, max_community: int, seed: int) -> nx.DiGraph:

  # 1. 無向LFRグラフを生成 (directed 引数は削除)
  G_undirected = nx.LFR_benchmark_graph(
      n,
      tau1,
      tau2,
      mu,
      average_degree=average_degree,
      min_community=min_community,
      max_community=max_community,
      seed=seed,
  )

  # 2. 有向グラフを作成し、ノードとその属性（community情報）を引き継ぐ
  G = nx.DiGraph()
  G.add_nodes_from(G_undirected.nodes(data=True))

  # 3. エッジに向きをつけて追加（50%の確率で u -> v または v -> u）
  rng = random.default_rng(seed)  # 乱数生成器の初期化
  for u, v in G_undirected.edges():
      if u == v:
          continue
      if rng.random() < 0.5:
          G.add_edge(u, v)
      else:
          G.add_edge(v, u)

  return G

In [ ]:

from pathlib import Path

name  = "LFR01"
n = 5000             # ノード数
tau1 = 3.0           # 次数分布の指数
tau2 = 1.016         # コミュニティサイズ分布の指数
mu = 0.8             # ミキシングパラメータ
average_degree = 10  # 平均次数
min_community = 100  # 最小コミュニティサイズ
max_community = 1500 # 最大コミュニティサイズ
seed = 6             # 乱数シード

G = generate_lfr_benchmark_graph(n, tau1, tau2, mu, average_degree, min_community, max_community, seed)
p = Path(f"../../data/{name}")
if not p.exists():
    p.mkdir(parents=True, exist_ok=True)
nx.write_edgelist(G, p.joinpath("edgelist.txt.gz"), data=False)
io.write_communitydict(G, p.joinpath("communities.txt.gz"))

### Community analysis

In [5]:
from collections import Counter
import numpy as np
import pandas as pd

competitor_scenarios_header_text = {
    "opponent_id": "Opponent ID",
    "opponent_size": "Opponent Size (規模)",
    "opponent_mean": "Opponent Mean (構造の強さ)",
    "inter_edges": "Inter Edges (パイプの太さ)",
    "target_bridges": "Target側ブリッジ数",
    "opponent_bridges": "Opponent側ブリッジ数",
    "target_gatekeeper": "Target側最大ゲートキーパーノード",
    "gatekeeper_edges": "その1人が持つ接続エッジ数"
}

def analyze_competitor_scenarios(G, communities, global_pr_scores, target_id=2):
    """
    Community target_id と他のコミュニティとの繋がり、および相手の構造特性を同時に解析する
    """
    if target_id >= len(communities):
        raise ValueError(f"target_id ({target_id}) がコミュニティ総数 ({len(communities)}) を超えています。")

    target_nodes = set(communities[target_id])
    scenario_data = []
    bridge_details = {}

    for i, comm_members in enumerate(communities):
        if i == target_id:
            continue

        current_nodes = set(comm_members)

        # 1. 境界エッジの抽出（パイプの太さ）
        inter_edges = list(nx.edge_boundary(G, target_nodes, current_nodes))
        edge_count = len(inter_edges)

        if edge_count == 0:
            continue  # 完全に独立しているコミュニティは除外

        # 2. 両側のブリッジノード（架け橋）を特定
        target_side_bridges = set(u for u, v in inter_edges)
        partner_side_bridges = set(v for u, v in inter_edges)

        # 3. 相手コミュニティ自体の「構造の強さ」を全体PRから計算
        partner_pr_values = [global_pr_scores[n] for n in comm_members if n in G]
        partner_mean = np.mean(partner_pr_values) if partner_pr_values else 0.0

        # 4. 最大のゲートキーパーの特定 (Target側)
        my_nodes_in_edges = [u for u, v in inter_edges]
        if my_nodes_in_edges:
            gatekeeper, gk_edges = Counter(my_nodes_in_edges).most_common(1)[0]
        else:
            gatekeeper, gk_edges = None, 0

        scenario_data.append({
            "opponent_id": i,
            "opponent_size": len(current_nodes),
            "opponent_mean": partner_mean,
            "inter_edges": edge_count,
            "target_bridges": len(target_side_bridges),
            "opponent_bridges": len(partner_side_bridges),
            "target_gatekeeper": gatekeeper,
            "gatekeeper_edges": gk_edges
        })

        bridge_details[i] = inter_edges

    df = pd.DataFrame(scenario_data)

    if not df.empty:
        df = df.sort_values(by="inter_edges", ascending=False).reset_index(drop=True)

    return df, bridge_details

In [8]:
import networkx as nx
from pathlib import Path

name = "LFR01"

path = Path("../../data") / name

# データ読み込み
G = nx.read_edgelist(path / "edgelist.txt.gz", create_using=nx.DiGraph(), nodetype=int)
communities = io.communitydict_to_communitylist(io.read_communitydict(path / "communities.txt.gz"))

# 全体 PageRank 計算
full_pr = nx.pagerank(G)

# 構造解析実行 (target_id=2)
scenarios_df, _ = analyze_competitor_scenarios(G, communities, full_pr, target_id=2)

print("📊 【コミュニティ2の競合シナリオ解析結果】")
scenarios_df

📊 【コミュニティ2の競合シナリオ解析結果】


,opponent_id,opponent_size,opponent_mean,inter_edges,target_bridges,opponent_bridges,target_gatekeeper,gatekeeper_edges
0,1,1082,0.000210,1726,799,871,4539,25
1,8,1159,0.000183,1689,795,878,4539,36
2,9,694,0.000178,860,542,474,4539,18
3,7,283,0.000173,301,243,190,4539,7
4,5,224,0.000190,263,231,146,2898,5
5,3,198,0.000205,249,213,138,4539,6
6,0,141,0.000184,187,157,102,4539,5
7,4,131,0.000171,142,122,83,328,3
8,6,128,0.000185,141,118,80,4539,4


In [9]:
import pandas as pd

# 2. 各コミュニティのPageRank集計
results = []
for i, comm_members in enumerate(communities):
    comm_scores = [full_pr[node] for node in comm_members if node in full_pr]

    results.append({
        "Community ID": i,
        "Size": len(comm_members),
        "PPR Sum": np.sum(comm_scores),
        "PPR Mean": np.mean(comm_scores)
    })

results_df = pd.DataFrame(results)

# 検算: 全コミュニティのPPR Sumの合計を表示 (1.0になるはず)
assert np.isclose(results_df["PPR Sum"].sum(), 1.0), "PPR Sumの合計が1.0ではありません。"

# PPR Sumが高い順にソートして表示
sorted_by_sum_df = results_df.sort_values(by="PPR Sum", ascending=False)
total_ppr_sum = results_df["PPR Sum"].sum()

print("📊 【コミュニティごとの PageRank 集計結果】")
sorted_by_sum_df

📊 【コミュニティごとの PageRank 集計結果】


,Community ID,Size,PPR Sum,PPR Mean
2,2,960,0.232270,0.000242
1,1,1082,0.227590,0.000210
8,8,1159,0.212209,0.000183
9,9,694,0.123631,0.000178
7,7,283,0.048970,0.000173
5,5,224,0.042662,0.000190
3,3,198,0.040554,0.000205
0,0,141,0.025994,0.000184
6,6,128,0.023657,0.000185
4,4,131,0.022463,0.000171


In [25]:
!cd ../../difftools && go run ./cmd/sample


Hello, World!
